<a href="https://colab.research.google.com/github/taeyeon0102/gdgoc-archive/blob/main/Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## <검증 세트 활용>

# 1. 데이터 불러오기, 정답 데이터 분리하기
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine_csv_data')

data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()
# 2. 훈련 세트, 테스트 세트 나누기
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)
# 3. 훈련 세트에서 검증 세트 분리하기
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target, test_size=0.2, random_state=42)
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [ ]:
# 4. 훈련
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))  #훈련 세트 평가
print(dt.score(val_input, val_target))  #검증 세트 평가

0.9971133028626413
0.864423076923077


In [ ]:
# 5. 교차 검증
from sklearn.model_selection import cross_validate
scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.01190376, 0.01939034, 0.01447248, 0.02838373, 0.02525067]), 'score_time': array([0.00170088, 0.00174212, 0.00169873, 0.00599551, 0.00179696]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [ ]:
# 위에서 얻은 scores의 평균을 구해서 최종 점수 산출
import numpy as np
print(np.mean(scores['test_score']))

0.855300214703487


In [ ]:
## 우리는 앞에서 훈련 세트과 테스트 세트를 분리할 때 데이터를 섞었다
## 하지만 만약 훈련 세트를 섞지 않아서 데이터를 섞은 후 교차검증을 하려면 ...
from sklearn.model_selection import StratifiedKFold  # <- 분류 모델의 경우 target class를 골고루 나누기 위해 StratifieKFold를 사용함
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)  # <- 분할기를 지정해야 함
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


In [ ]:
## <GridSearchCV>
# >> 1개 하이퍼파라미터
# 1. 그리드 서치 객체 만들기
from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]} #  <- min_impurity_decrease의 최적값 찾기 (0.0001~0.0005)
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1) # <- n_jobs는 사용할 cpu수 지정(기본값:1, -1은 모든 cpu 사용)
# 2. 그리드 서치 실행 (기본값: 5번, 즉 하이퍼파라미터 각 값에 대해서 5-fold cross validation 수행)
gs.fit(train_input, train_target)
# 3. 최적의 하이퍼파라미터
dt = gs.best_estimator_
print(dt.score(train_input, train_target))
print(gs.best_params_)  # <- 최적의 값
print(gs.cv_results_['mean_test_score'])  # <- 각 하이퍼파라미터의 최종 검증 점수 (5번 교차검증)

best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index]) # <- 각 하이퍼파리미터의 최종 검증 점수 중 최대 고르기

0.9615162593804117
{'min_impurity_decrease': 0.0001}
[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]
{'min_impurity_decrease': 0.0001}


In [ ]:
# >> 3개 하이퍼파라미터
# 1. 하이퍼파라미터: 불순도 감소 최소량, 트리의 깊이 제한, 노드를 나누기 위한 최소 샘플 수
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
          'max_depth' : range(5, 20, 1),
          'min_samples_split' : range(2, 100, 10)}

# 2. 그리드 서치 실행
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

# 3. 최적의 하이퍼파라미터
print(gs.best_params_)

print(np.max(gs.cv_results_['mean_test_score'])) # <- 최적의 하이퍼파라미터의 교차 검증 점수

{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}
0.8683865773302731


In [ ]:
## <RandomSearchCV>
# 1. 탐색할 파라미터 지정
from scipy.stats import uniform, randint # <- 균등하게 무작위 값 산출
from sklearn.model_selection import RandomizedSearchCV
params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': range(20, 50),
          'min_samples_split': range(2, 25),
          'min_samples_leaf': randint(1, 25)}
# 2. 랜덤 서치 실행
from sklearn.model_selection import RandomizedSearchCV
gs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42, splitter = 'random'), params, n_iter=100, n_jobs=-1, random_state=42)
gs.fit(train_input, train_target)
# 3. 결과 확인
print(gs.best_params_)
print(np.max(gs.cv_results_['mean_test_score']))
# 4. 최종 모델의 테스트 세트 평가
dt = gs.best_estimator_
print(dt.score(test_input, test_target))

{'max_depth': 43, 'min_impurity_decrease': np.float64(0.00011407982271508446), 'min_samples_leaf': 19, 'min_samples_split': 18}
0.8458726956392981
0.786923076923077
